# 02 — Cross-Validation Architectures and Leak Prevention

**Session:** 2026-11-06 · Design Validation  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

---

## Overview

In real-world data science, cross-validation is not just a mechanism to score models; it is the **computational blueprint** that guarantees models generalize to unseen test distributions without leaking future or grouped information.

This notebook builds and audits five essential validation split topologies from first principles:
1. **K-Fold Cross Validation:** Standard randomized partitioner with out-of-fold estimation.
2. **Stratified K-Fold:** Label distribution preservation for imbalanced targets.
3. **GroupKFold:** Preventing intra-cluster identity leakage across longitudinal patient visits.
4. **TimeSeriesSplit:** Walk-forward validation enforcing strict temporal causality without lookahead bias.
5. **Adversarial Validation:** Training a domain discriminator to audit train-test covariate distribution shift.

In [ ]:
import csv
from pathlib import Path

import numpy as np

np.random.seed(42)
print("Environment initialized with deterministic PRNG seed.")

## 1. K-Fold Partitioning from First Principles

Standard K-Fold cross validation partitions a dataset of $N$ samples into $K$ disjoint folds of size $\lfloor N/K \rfloor$. In each fold $k \in \{1, \dots, K\}$, fold $k$ serves as the validation set while the remaining $K-1$ folds constitute the training set.

In [ ]:
def k_fold_split(
    n_samples: int,
    n_splits: int = 5,
    shuffle: bool = True,
    seed: int = 42,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Generate K-Fold train and validation index splits from scratch."""
    indices = np.arange(n_samples)
    if shuffle:
        rng = np.random.default_rng(seed)
        rng.shuffle(indices)

    fold_sizes = np.full(n_splits, n_samples // n_splits, dtype=int)
    fold_sizes[: n_samples % n_splits] += 1
    current = 0
    splits = []
    for fold_size in fold_sizes:
        start, stop = current, current + fold_size
        val_idx = indices[start:stop]
        train_idx = np.concatenate([indices[:start], indices[stop:]])
        splits.append((train_idx, val_idx))
        current = stop
    return splits


# Verification on 20 samples with 5 folds
toy_splits = k_fold_split(n_samples=20, n_splits=5, shuffle=False)
for k, (tr, val) in enumerate(toy_splits):
    print(f"Fold {k + 1}: Train {len(tr)} samples | Val {len(val)} samples -> {val}")

## 2. Stratified K-Fold for Imbalanced Classification

When predicting rare events (such as customer churn, fraud, or rare medical diagnoses), naive randomized splitting can create significant fold-to-fold variance in the positive class prevalence.

**Stratified K-Fold** partitions the dataset separately within each class label $y \in \mathcal{Y}$, ensuring that every training and validation fold mirrors the global class prior $P(Y=c)$.

In [ ]:
def stratified_k_fold_split(
    labels: np.ndarray,
    n_splits: int = 5,
    shuffle: bool = True,
    seed: int = 42,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Generate Stratified K-Fold splits preserving target distribution across folds."""
    rng = np.random.default_rng(seed)
    class_indices: dict[int, np.ndarray] = {}
    for cls in np.unique(labels):
        idx = np.where(labels == cls)[0]
        if shuffle:
            rng.shuffle(idx)
        class_indices[cls] = idx

    # Distribute class indices evenly into K folds
    folds: list[list[int]] = [[] for _ in range(n_splits)]
    for _cls, idxs in class_indices.items():
        for i, idx in enumerate(idxs):
            folds[i % n_splits].append(idx)

    all_indices = set(range(len(labels)))
    splits = []
    for k in range(n_splits):
        val_idx = np.array(folds[k], dtype=int)
        train_idx = np.array(list(all_indices - set(val_idx)), dtype=int)
        splits.append((train_idx, val_idx))
    return splits


# Load customer churn dataset
churn_path = Path("../data/customer_churn_stratified.csv")
with churn_path.open("r", encoding="utf-8") as fp:
    reader = csv.DictReader(fp)
    churn_records = list(reader)

y_churn = np.array([int(r["churn"]) for r in churn_records])
print(f"Dataset samples: {len(y_churn)} | Overall churn rate: {np.mean(y_churn):.4f}")

# Compare naive vs stratified K-Fold
naive_splits = k_fold_split(len(y_churn), n_splits=5, shuffle=True, seed=10)
strat_splits = stratified_k_fold_split(y_churn, n_splits=5, shuffle=True, seed=10)

print("\nNaive K-Fold Validation Churn Rates:")
for k, (_, v_idx) in enumerate(naive_splits):
    print(f"  Fold {k + 1}: {np.mean(y_churn[v_idx]):.4f}")

print("\nStratified K-Fold Validation Churn Rates:")
for k, (_, v_idx) in enumerate(strat_splits):
    print(f"  Fold {k + 1}: {np.mean(y_churn[v_idx]):.4f}")

## 3. GroupKFold: Preventing Subject Identity Data Leakage

In medical imaging, speech recognition, and IoT applications, multiple rows in a dataset often belong to the same underlying entity (e.g., patient, speaker, device).

If visits from `PAT_0001` are randomly scattered between train and validation splits:
- The model memorizes patient-specific biological baselines.
- The validation score is artificially inflated.
- The model collapses in production when evaluating a completely new patient!

`GroupKFold` guarantees that **all records for any given group appear strictly in either the training set OR the validation set**, never both.

In [ ]:
def group_k_fold_split(
    groups: np.ndarray,
    n_splits: int = 5,
    seed: int = 42,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Partition indices such that unique groups do not overlap between train and val."""
    unique_groups = np.unique(groups)
    rng = np.random.default_rng(seed)
    rng.shuffle(unique_groups)

    # Assign groups to folds
    group_folds: list[set] = [set() for _ in range(n_splits)]
    for i, grp in enumerate(unique_groups):
        group_folds[i % n_splits].add(grp)

    splits = []
    all_idx = np.arange(len(groups))
    for k in range(n_splits):
        val_groups = group_folds[k]
        val_mask = np.isin(groups, list(val_groups))
        val_idx = all_idx[val_mask]
        train_idx = all_idx[~val_mask]
        splits.append((train_idx, val_idx))
    return splits


# Load clinical patient records
clin_path = Path("../data/clinical_patient_records.csv")
with clin_path.open("r", encoding="utf-8") as fp:
    clin_records = list(csv.DictReader(fp))

patient_groups = np.array([r["patient_id"] for r in clin_records])
print(
    f"Total records: {len(patient_groups)} across {len(np.unique(patient_groups))} distinct patients."
)

grp_splits = group_k_fold_split(patient_groups, n_splits=5, seed=42)

for k, (tr_idx, val_idx) in enumerate(grp_splits):
    tr_pats = set(patient_groups[tr_idx])
    val_pats = set(patient_groups[val_idx])
    overlap = tr_pats.intersection(val_pats)
    print(
        f"Fold {k + 1}: Train rows {len(tr_idx):3d} ({len(tr_pats):2d} pats) | "
        f"Val rows {len(val_idx):3d} ({len(val_pats):2d} pats) | Leakage Overlap: {len(overlap)}"
    )
    assert len(overlap) == 0, "Data leakage detected: patient in both train and val!"
print("GroupKFold integrity verified: zero subject leakage across all folds.")

## 4. TimeSeriesSplit: Causal Walk-Forward Validation

Time series and financial telemetry require validation strategies that strictly respect the arrow of time:
$$\max(t_{\text{train}}) < \min(t_{\text{val}})$$

Random splitting in temporal domains introduces **lookahead bias**, where future trends leak into past predictions.

In [ ]:
def time_series_split(
    n_samples: int,
    n_splits: int = 5,
    min_train_size: int | None = None,
) -> list[tuple[np.ndarray, np.ndarray]]:
    """Generate expanding walk-forward train and validation index splits."""
    test_size = n_samples // (n_splits + 1)
    min_train = min_train_size or test_size
    splits = []
    for k in range(n_splits):
        train_end = min_train + k * test_size
        val_end = min_train + (k + 1) * test_size
        train_idx = np.arange(0, train_end)
        val_idx = np.arange(train_end, min(val_end, n_samples))
        splits.append((train_idx, val_idx))
    return splits


# Load sensor telemetry
ts_path = Path("../data/sensor_telemetry_timeseries.csv")
with ts_path.open("r", encoding="utf-8") as fp:
    ts_records = list(csv.DictReader(fp))

ts_splits = time_series_split(n_samples=len(ts_records), n_splits=5)
for k, (tr, val) in enumerate(ts_splits):
    tr_start, tr_end = ts_records[tr[0]]["timestamp"], ts_records[tr[-1]]["timestamp"]
    v_start, v_end = ts_records[val[0]]["timestamp"], ts_records[val[-1]]["timestamp"]
    print(
        f"Split {k + 1}: Train [{tr_start} -> {tr_end}] ({len(tr):3d} samples) | "
        f"Val [{v_start} -> {v_end}] ({len(val):3d} samples)"
    )

## 5. Adversarial Validation: Auditing Covariate Shift

When public leaderboard scores diverge from local validation scores, the primary culprit is often **covariate shift** ($P_{\text{train}}(X) \neq P_{\text{test}}(X)$).

**Adversarial Validation Algorithm:**
1. Concatenate train features $X_{\text{train}}$ and test features $X_{\text{test}}$.
2. Create an adversarial label: $z_i = 0$ if sample came from train, $z_i = 1$ if test.
3. Train a binary classifier to predict $z_i$ from $X$.
4. Evaluate ROC-AUC:
   - If $\text{AUC} \approx 0.50$: Train and test distributions are identical. Local CV will reliably predict test performance.
   - If $\text{AUC} > 0.75$: Severe covariate shift exists! The classifier easily distinguishes train from test.
5. In severe shift scenarios, practitioner remedies include:
   - Dropping features with the highest adversarial feature importance.
   - Re-weighting training samples using odds ratios: $w(x) = \frac{P(z=1|x)}{P(z=0|x)}$.
   - Selecting a validation set whose samples most closely resemble the test distribution ($P(z=1|x) \approx 1$).

In [ ]:
# Load covariate shift benchmark
cov_path = Path("../data/covariate_shift_benchmark.csv")
with cov_path.open("r", encoding="utf-8") as fp:
    cov_records = list(csv.DictReader(fp))

X_cov = np.array([[float(r["feature_1"]), float(r["feature_2"])] for r in cov_records])
# Adversarial target: 0 for train, 1 for test
z_adversarial = np.array([1 if r["split_origin"] == "test" else 0 for r in cov_records])


# Compute logistic regression discriminator manually using gradient descent
def sigmoid(v: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(v, -15.0, 15.0)))


weights = np.zeros(2)
bias = 0.0
lr = 0.05

for _ in range(300):
    logits = np.dot(X_cov, weights) + bias
    preds = sigmoid(logits)
    err = preds - z_adversarial
    dw = np.dot(X_cov.T, err) / len(X_cov)
    db = np.mean(err)
    weights -= lr * dw
    bias -= lr * db

prob_test = sigmoid(np.dot(X_cov, weights) + bias)


# Compute ROC-AUC via Wilcoxon rank-sum statistic
def compute_roc_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    pos_scores = y_score[y_true == 1]
    neg_scores = y_score[y_true == 0]
    n_pos = len(pos_scores)
    n_neg = len(neg_scores)
    # Count pairs where pos > neg
    all_scores = np.concatenate([pos_scores, neg_scores])
    ranks = np.argsort(np.argsort(all_scores)) + 1
    rank_sum_pos = np.sum(ranks[:n_pos])
    u_stat = rank_sum_pos - (n_pos * (n_pos + 1)) / 2.0
    return float(u_stat / (n_pos * n_neg))


adversarial_auc = compute_roc_auc(z_adversarial, prob_test)
print(f"Adversarial Discriminator ROC-AUC: {adversarial_auc:.4f}")

if adversarial_auc > 0.70:
    print("Diagnosis: Severe covariate distribution shift detected between train and test sets!")
    print("Action: Local random validation will NOT reliably mirror test performance.")
else:
    print("Diagnosis: Train and test distributions are statistically aligned.")

## Summary: Validation Topology Selection Matrix

| Scenario | Recommended Strategy | Failure Mode if Violated |
| :--- | :--- | :--- |
| Standard i.i.d. Tabular | $K$-Fold Cross Validation ($K=5$ or $10$) | High estimation variance if $K$ too small; computational bottleneck if $K$ too large |
| Imbalanced Binary / Multi-class | Stratified $K$-Fold | Some folds lack positive examples entirely, causing infinite loss or NaN metrics |
| Clustered / Longitudinal Entities | GroupKFold (grouped by entity ID) | Data leakage: model memorizes entity identifiers and overfits |
| Temporal / Time Series Data | TimeSeriesSplit (Walk-Forward) | Lookahead bias: future information bleeds into historical training |
| Domain / Distribution Shift | Adversarial Validation + Domain Adaptation | Local CV score completely decouples from test / leaderboard performance |